# آموزش تشخیص تصادف در Colab؛ ۳۰۰ ویدئو

**معماری: YOLOv8n تا P4 → LSTM معمولی → Attention → Sigmoid**؛ تعداد پارامترها: **406,354**. معماری همان نسخهٔ قبلی است. وزن آماده‌ای از اینترنت دریافت نمی‌شود؛ شروع تازه از صفر است. اگر checkpoint همین اجرای خودتان روی Drive موجود باشد، `resume=True` آن را ادامه می‌دهد.

ابتدا پری‌پراسس ۳۰۰تایی باید پیام `PREPROCESSING COMPLETED | OK` بدهد. در Colab از مسیر **Runtime → Change runtime type**، یک **GPU** انتخاب کنید، سپس سلول‌های این نوت‌بوک را از بالا اجرا کنید و اتصال Drive را تأیید کنید. لازم نیست نوت‌بوک پری‌پراسس در همین kernel اجرا شده باشد یا فایلش کنار مدل باشد.

ورودی دقیق: `data/preprocessed_300_0p2s_224`، فایل‌های `train_240.csv` و `val_60.csv`، شامل **۲۴۰ آموزش (۱۲۰/۱۲۰) و ۶۰ اعتبارسنجی (۳۰/۳۰)**. فریم‌ها RGB / uint8 / 224×224 با فاصلهٔ ۰٫۲ ثانیه، یعنی ۵fps هستند. تمام فریم‌های آماده حفظ می‌شوند.

هیچ دانلود ویدئو، استخراج فریم، resize، نمونه‌گیری زمانی یا تقسیم داده تکرار نمی‌شود. Dataset و آگمنتیشن همان تعریف‌های پری‌پراسس هستند و داخل این فایل قرار دارند؛ مقیاس‌کردن پیکسل‌ها و آگمنتیشن آنلاین یک بار هنگام بارگذاری انجام می‌شوند.

برای سرعت خواندن، فقط NPZها و دو CSV لازم از Drive به دیسک محلی Colab کپی می‌شوند؛ ویدئوهای اصلی کپی نمی‌شوند. وزن‌های `best.pt` و `last.pt`، تاریخچه و نمودارها روی Drive ذخیره می‌شوند. کپی محلی داده‌ها بعد از حذف runtime دوباره لازم است؛ پری‌پراسس دوباره اجرا نمی‌شود.

خروجی عمومی مدل sigmoid است؛ loss از logit خام با `BCEWithLogitsLoss` محاسبه می‌شود. انتخاب وزن، AdamW، scheduler، آستانهٔ ۰٫۵، LSTM یک‌طرفه و تنظیمات اصلی آموزش همان نسخهٔ قبلی هستند. این نوت‌بوک نتیجهٔ آموزش واقعی روی ویدئوها ندارد.

**بررسی نسخه:** اجرای کامل با دادهٔ مصنوعی و شبیه‌سازی Drive/Colab روی CPU بررسی شده است؛ کپی ورودی، خروجی sigmoid، حفظ فریم‌ها، گرادیان‌ها، padding، ذخیره/بارگذاری وزن، همهٔ نمودارها و ادامهٔ آموزش پس از حذف runtime آزمایش شدند. در همان محیط CPU، ادامه از checkpoint همان وزن‌های اجرای پیوسته را تولید کرد. همچنین مسیر ورودی با manifest واقعیِ ۲۴۰/۶۰ مصنوعی و ۳۰۰ NPZ بررسی شد. آموزش روی ویدئوهای واقعی، اتصال زندهٔ Google و عملکرد واقعی GPU آزمایش نشده‌اند.


## ۱. وابستگی‌ها

کتابخانه‌های موجود Colab حفظ می‌شوند؛ سلول زیر فقط وابستگی‌های نصب‌نشده را نصب می‌کند. نصب مجدد PyTorch برای محیط آمادهٔ Colab انجام نمی‌شود.

In [1]:
import importlib.util
import subprocess
import sys

requirements = {
    "torch": "torch", "numpy": "numpy", "pandas": "pandas",
    "matplotlib": "matplotlib", "sklearn": "scikit-learn",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready; existing PyTorch/CUDA installation preserved.")


Dependencies ready; existing PyTorch/CUDA installation preserved.


In [2]:
from pathlib import Path
import random
import time
import shutil
import hashlib
import json
from contextlib import contextmanager, nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torch.nn.utils.rnn import (
    PackedSequence, pack_padded_sequence,
    pad_packed_sequence, pad_sequence,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
)

## اتصال Google Drive

حسابی را انتخاب کنید که خروجی پری‌پراسس در آن ذخیره شده است.

In [4]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## ۲. مسیرها و تنظیمات

`project_dir` باید همان پوشهٔ پروژه در پری‌پراسس باشد؛ پیش‌فرض با فایل جدید هماهنگ است. `copy_to_local=True` باعث خواندن فریم‌ها از دیسک محلی می‌شود. `out_dir` برای این مدل و تعداد ۳۰۰ جداست.

با `resume=True`، اجرای دوباره از آخرین epoch ذخیره‌شدهٔ همین اجرا ادامه می‌یابد. اگر قصد آموزش تازه با تنظیمات دیگر دارید، `out_dir` تازه انتخاب کنید. پیش‌فرض `require_cuda=True` جلوی شروع تصادفی آموزش طولانی روی CPU را می‌گیرد.

batch_size=1، تجمیع گرادیان در ۸ گام و پردازش CNN در دسته‌های ۸ فریم حفظ شده‌اند. checkpointing حافظهٔ مصرفی را با محاسبهٔ مجدد کمتر می‌کند و هیچ فریمی حذف نمی‌شود.


In [5]:
from pathlib import Path
import random

import numpy as np
import torch


CFG = {
    "project_dir": "/content/drive/MyDrive/DL-Project/image_neural_network",

    # Keep the existing folder to reuse prepared frames.
    "frame_dir": "data/preprocessed_300_0p2s_224",
    "train_csv": "train_340.csv",
    "val_csv": "val_60.csv",
    "test_csv": "test_30.csv",

    "expected_train": 340,
    "expected_val": 60,
    "expected_test": 30,

    "out_dir": "data/yolo_lstm_v2_430_f1",

    "seed": 42,
    "batch_size": 1,
    "accum_steps": 8,
    "workers": 0,
    "frame_chunk": 8,
    "checkpoint_cnn": True,

    "hidden_size": 64,
    "attention_size": 32,
    "head_size": 32,
    "feature_dropout": 0.2,
    "dropout": 0.3,

    "epochs": 15,
    "min_epochs": 15,
    "early_patience": 8,

    "lr": 0.0003,
    "weight_decay": 0.0001,
    "grad_clip": 1.0,

    "lr_factor": 0.5,
    "lr_patience": 2,
    "min_lr": 1e-6,

    # Fixed threshold for comparable training logs.
    "threshold": 0.5,

    # Validation-only selection of checkpoint and threshold.
    "selection_rule": "val_f1_grid_v1",
    "threshold_grid": [
        0.20, 0.25, 0.30, 0.35, 0.40,
        0.45, 0.50, 0.55, 0.60, 0.65,
        0.70, 0.75, 0.80,
    ],

    # Reuse the previous local cache.
    "local_project_dir": "/content/accident_training_300",
    "copy_to_local": True,
    "require_cuda": True,

    # Fresh run. Use True only when continuing THIS run.
    "resume": True,
    "log_every": 10,
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])

PROJECT_DIR = Path(CFG["project_dir"]).resolve()
LOCAL_PROJECT_DIR = Path(CFG["local_project_dir"]).resolve()
LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FRAME_DIR = PROJECT_DIR / CFG["frame_dir"]
SOURCE_TRAIN_CSV = SOURCE_FRAME_DIR / CFG["train_csv"]
SOURCE_VAL_CSV = SOURCE_FRAME_DIR / CFG["val_csv"]
SOURCE_TEST_CSV = SOURCE_FRAME_DIR / CFG["test_csv"]

for required_path in [
    SOURCE_FRAME_DIR,
    SOURCE_TRAIN_CSV,
    SOURCE_VAL_CSV,
    SOURCE_TEST_CSV,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Prepared input not found: {required_path}\n"
            "Complete incremental preprocessing first: "
            "Train 340 / Validation 60 / Test 30."
        )

FRAME_DIR = (
    LOCAL_PROJECT_DIR / CFG["frame_dir"]
    if CFG["copy_to_local"]
    else SOURCE_FRAME_DIR
)

TRAIN_CSV = FRAME_DIR / CFG["train_csv"]
VAL_CSV = FRAME_DIR / CFG["val_csv"]

# Test is read from Drive only during final evaluation.
TEST_CSV = SOURCE_TEST_CSV

OUT = PROJECT_DIR / CFG["out_dir"]
OUT.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device, "| PyTorch:", torch.__version__)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif CFG["require_cuda"]:
    raise RuntimeError(
        "GPU is not available. Select a GPU runtime before training."
    )

if CFG["workers"] != 0:
    raise ValueError("Keep workers=0 for this notebook workflow.")

print("Prepared source on Drive:", SOURCE_FRAME_DIR)
print("Training reads:", FRAME_DIR)
print("Final test source:", TEST_CSV)
print("Persistent training outputs:", OUT)
print(
    f"Expected videos: train={CFG['expected_train']}, "
    f"validation={CFG['expected_val']}, "
    f"test={CFG['expected_test']}"
)

Device: cuda | PyTorch: 2.11.0+cu130
GPU: Tesla T4
Prepared source on Drive: /content/drive/MyDrive/DL-Project/image_neural_network/data/preprocessed_300_0p2s_224
Training reads: /content/accident_training_300/data/preprocessed_300_0p2s_224
Final test source: /content/drive/MyDrive/DL-Project/image_neural_network/data/preprocessed_300_0p2s_224/test_30.csv
Persistent training outputs: /content/drive/MyDrive/DL-Project/image_neural_network/data/yolo_lstm_v2_430_f1
Expected videos: train=340, validation=60, test=30


### آماده‌کردن ورودی محلی

این مرحله فقط کپی فایل‌های آماده است. گزارش پیشرفت دارد؛ کپی موجود با اندازه و زمان ویرایش یکسان استفاده می‌شود. کل بررسی سنگین پری‌پراسس تکرار نمی‌شود.

In [8]:
def relative_frame_path(video_path):
    relative = Path(video_path).with_suffix(".npz")
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("Expected a relative video path in the manifest.")
    return relative


source_splits = {}
signature_rows = {}
relative_paths = [Path(CFG["train_csv"]), Path(CFG["val_csv"])]
for name, csv_path, expected in [
    ("train", SOURCE_TRAIN_CSV, CFG["expected_train"]),
    ("val", SOURCE_VAL_CSV, CFG["expected_val"]),
]:
    frame = pd.read_csv(csv_path, dtype={"video_path": str})
    if not {"video_path", "label"}.issubset(frame.columns):
        raise ValueError(f"{name}: missing video_path or label")
    if len(frame) != expected or not frame["video_path"].is_unique:
        raise ValueError(f"{name}: expected {expected} unique videos, got {len(frame)}")
    if frame["label"].value_counts().to_dict() != {0: expected // 2, 1: expected // 2}:
        raise ValueError(f"{name}: expected an even, balanced binary split")
    source_splits[name] = frame
    signature_rows[name] = [[str(p), int(y)] for p, y in frame[["video_path", "label"]].itertuples(index=False, name=None)]
    relative_paths.extend(relative_frame_path(p) for p in frame["video_path"])
if not set(source_splits["train"]["video_path"]).isdisjoint(source_splits["val"]["video_path"]):
    raise ValueError("Train and validation overlap")
data_signature = hashlib.sha256(json.dumps(signature_rows, sort_keys=True).encode()).hexdigest()
missing = [p for p in relative_paths if not (SOURCE_FRAME_DIR / p).is_file()]
if missing:
    raise FileNotFoundError(f"Missing prepared input: {SOURCE_FRAME_DIR / missing[0]}")

if CFG["copy_to_local"]:
    FRAME_DIR.mkdir(parents=True, exist_ok=True)
    pending = []
    for relative in relative_paths:
        source, target = SOURCE_FRAME_DIR / relative, FRAME_DIR / relative
        src_stat = source.stat()
        if target.is_file():
            dst_stat = target.stat()
            if dst_stat.st_size == src_stat.st_size and dst_stat.st_mtime_ns == src_stat.st_mtime_ns:
                continue
        pending.append((source, target))
    required_bytes = sum(source.stat().st_size for source, _ in pending)
    free_bytes = shutil.disk_usage(LOCAL_PROJECT_DIR).free
    print(f"Local copy: {len(pending)} files pending | {required_bytes / 1024**3:.2f} GiB")
    if required_bytes > free_bytes:
        raise OSError("Not enough local Colab disk space for prepared data.")
    for i, (source, target) in enumerate(pending, start=1):
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + ".copying")
        try:
            shutil.copy2(source, temporary)
            if temporary.stat().st_size != source.stat().st_size:
                raise IOError(f"Incomplete local copy: {target}")
            temporary.replace(target)
        except BaseException:
            temporary.unlink(missing_ok=True)
            raise
        if i == 1 or i % 25 == 0 or i == len(pending):
            print(f"  Copied {i}/{len(pending)} files", flush=True)
print("Prepared inputs ready; no download, frame extraction, resize or resampling performed.")


Local copy: 399 files pending | 6.70 GiB
  Copied 1/399 files
  Copied 25/399 files
  Copied 50/399 files
  Copied 75/399 files
  Copied 100/399 files
  Copied 125/399 files
  Copied 150/399 files
  Copied 175/399 files
  Copied 200/399 files
  Copied 225/399 files
  Copied 250/399 files
  Copied 275/399 files
  Copied 300/399 files
  Copied 325/399 files
  Copied 350/399 files
  Copied 375/399 files
  Copied 399/399 files
Prepared inputs ready; no download, frame extraction, resize or resampling performed.


## ۳. Dataset و آگمنتیشن قبلی

همان تعریف‌های پری‌پراسس داخل این فایل قرار گرفته‌اند؛ فایل `.ipynb` دیگری خوانده یا اجرا نمی‌شود. NPZ خام آماده دارای پیکسل‌های ۰ تا ۲۵۵ است؛ تبدیل به float در بازهٔ ۰ تا ۱ همان‌جا انجام می‌شود. آگمنتیشن فقط در آموزش فعال است و تغییر روشنایی/کنتراست در تمام فریم‌های یک ویدئو یکسان است. مسیرها و متادیتای محیطی وارد شبکه نمی‌شوند.


In [9]:
def video_augmentation(frames, probability=0.5):
    if np.random.random() >= probability:
        return frames

    # Apply the same adjustment to every frame of one video.
    brightness = np.random.uniform(0.9, 1.1)
    contrast = np.random.uniform(0.9, 1.1)

    frames = (frames - 0.5) * contrast + 0.5
    frames = frames * brightness
    frames = np.clip(frames, 0.0, 1.0)

    return frames.astype(np.float32)

class AccidentVideoDataset(Dataset):

    def __init__(self, csv_path, frame_dir, augment=False):
        self.df = pd.read_csv(csv_path)
        self.frame_dir = Path(frame_dir)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        frame_path = (
            self.frame_dir
            / Path(row["video_path"]).with_suffix(".npz")
        )

        with np.load(frame_path, allow_pickle=False) as data:
            frames = data["frames"]

        # Normalize pixels from [0, 255] to [0, 1].
        frames = frames.astype(np.float32) / 255.0

        if self.augment:
            frames = video_augmentation(frames)

        # [T, H, W, C] -> [T, C, H, W]
        frames = torch.from_numpy(
            np.ascontiguousarray(frames)
        ).permute(0, 3, 1, 2).contiguous()

        label = torch.tensor(
            int(row["label"]),
            dtype=torch.long
        )

        return frames, label


def collate_videos(batch):
    videos, labels = zip(*batch)
    lengths = torch.tensor([len(v) for v in videos], dtype=torch.long)
    padded = pad_sequence(videos, batch_first=True, padding_value=0.0)
    return padded, lengths, torch.stack(labels).float()


In [10]:
train_ds = AccidentVideoDataset(TRAIN_CSV, FRAME_DIR, augment=True)
train_eval_ds = AccidentVideoDataset(TRAIN_CSV, FRAME_DIR, augment=False)
val_ds = AccidentVideoDataset(VAL_CSV, FRAME_DIR, augment=False)

for name, ds, expected in [
    ("Train", train_ds, CFG["expected_train"]),
    ("Validation", val_ds, CFG["expected_val"]),
]:
    if len(ds) != expected:
        raise ValueError(f"{name}: expected {expected} videos, got {len(ds)}")
    if not ds.df["video_path"].is_unique:
        raise ValueError(f"{name}: duplicate video paths")
    if not ds.df["label"].isin([0, 1]).all() or ds.df["label"].nunique() != 2:
        raise ValueError(f"{name}: labels must contain both 0 and 1")
    paths = [FRAME_DIR / Path(p).with_suffix(".npz") for p in ds.df["video_path"]]
    missing = [str(p) for p in paths if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"{name}: {len(missing)} NPZ files missing; first: {missing[0]}")
    print(name, len(ds), ds.df["label"].value_counts().to_dict())

if not set(train_ds.df["video_path"]).isdisjoint(set(val_ds.df["video_path"])):
    raise ValueError("Train and validation overlap")

generator = torch.Generator().manual_seed(CFG["seed"])
loader_kwargs = {
    "batch_size": CFG["batch_size"],
    "num_workers": CFG["workers"],
    "collate_fn": collate_videos,
    "pin_memory": device.type == "cuda",
}
train_loader = DataLoader(train_ds, shuffle=True, generator=generator, **loader_kwargs)
train_eval_loader = DataLoader(train_eval_ds, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

sample_video, sample_label = train_eval_ds[0]
print("Prepared sample [T, C, H, W]:", tuple(sample_video.shape))
print("Pixel range after YOUR Dataset:", float(sample_video.min()), float(sample_video.max()))
print("All prepared frames are retained. Source interval: 0.2 seconds.")
del sample_video

Train 340 {0: 170, 1: 170}
Validation 60 {1: 30, 0: 30}
Prepared sample [T, C, H, W]: (201, 3, 224, 224)
Pixel range after YOUR Dataset: 0.0 1.0
All prepared frames are retained. Source interval: 0.2 seconds.


## ۴. استخراج ویژگی YOLOv8n تا P4 و LSTM معمولی

| مرحله | ساختار | خروجی هر فریم برای ورودی ۲۲۴ |
|---|---|---|
| ۰ | Conv 3→16، stride 2 | 16×112×112 |
| ۱ | Conv 16→32، stride 2 | 32×56×56 |
| ۲ | C2f، 32 کانال، یک bottleneck | 32×56×56 |
| ۳ | Conv 32→64، stride 2 | 64×28×28 |
| ۴ | C2f، 64 کانال، دو bottleneck | 64×28×28 |
| ۵ | Conv 64→128، stride 2 | 128×14×14 |
| ۶ | C2f، 128 کانال، دو bottleneck | 128×14×14 |
| خلاصه‌سازی | Global Average Pooling | 128 |
| زمان | LSTM یک‌لایه و یک‌طرفه | 64 |
| Attention | Linear 64→32، Tanh، Linear 32→1، softmax زمانی | یک بردار 64تایی برای ویدئو |
| طبقه‌بند | LayerNorm، Linear 64→32، ReLU، Dropout 0.3، Linear 32→1 | یک logit |
| خروجی | Sigmoid | احتمال کلاس ۱ |

بخش تصویری از ساختار استاندارد YOLOv8n تا P4 پیروی می‌کند: Conv+BatchNorm+SiLU و C2f با shortcut. شاخه‌های P5، SPPF، neck و detection head در این مدل وجود ندارند. این تغییر تعداد پارامترها را محدود می‌کند؛ تعداد دقیق در سلول بعد چاپ می‌شود.

Checkpointing با محاسبهٔ مجدد بخشی از CNN حافظه را کاهش می‌دهد. در محاسبهٔ مجدد، آمار running مربوط به BatchNorm دوباره به‌روزرسانی نمی‌شود.

منابع ساختار تصویری (Ultralytics، AGPL-3.0):
- [YOLOv8 YAML](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/models/v8/yolov8.yaml)
- [C2f و Bottleneck](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/nn/modules/block.py)
- [Conv](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/nn/modules/conv.py)

بلوک‌های زیر پیاده‌سازی PyTorch سازگار با بخش انتخاب‌شده‌اند؛ وزن‌ها کاملاً تصادفی مقداردهی می‌شوند.

In [11]:
import torch
import torch.nn as nn

from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torch.nn.utils.rnn import (
    PackedSequence,
    pack_padded_sequence,
    pad_packed_sequence,
)


class YoloConv(nn.Module):
    # Conv + GroupNorm + SiLU
    def __init__(self, c1, c2, kernel=1, stride=1):
        super().__init__()

        self.conv = nn.Conv2d(
            c1, c2,
            kernel_size=kernel,
            stride=stride,
            padding=kernel // 2,
            bias=False,
        )
        self.bn = nn.GroupNorm(
            num_groups=8,
            num_channels=c2,
            eps=1e-3,
        )
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class YoloBottleneck(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.cv1 = YoloConv(channels, channels, 3)
        self.cv2 = YoloConv(channels, channels, 3)

    def forward(self, x):
        return x + self.cv2(self.cv1(x))


class YoloC2f(nn.Module):
    def __init__(self, c1, c2, repeats):
        super().__init__()

        hidden = c2 // 2

        self.cv1 = YoloConv(c1, 2 * hidden, 1)
        self.blocks = nn.ModuleList(
            YoloBottleneck(hidden) for _ in range(repeats)
        )
        self.cv2 = YoloConv((2 + repeats) * hidden, c2, 1)

    def forward(self, x):
        parts = list(self.cv1(x).chunk(2, dim=1))

        for block in self.blocks:
            parts.append(block(parts[-1]))

        return self.cv2(torch.cat(parts, dim=1))


class YoloLSTMClassifier(nn.Module):
    def __init__(
        self,
        hidden_size=64,
        attention_size=32,
        head_size=32,
        feature_dropout=0.2,
        dropout=0.3,
        frame_chunk=8,
        checkpoint_cnn=True,
    ):
        super().__init__()

        self.frame_chunk = frame_chunk
        self.checkpoint_cnn = checkpoint_cnn

        self.backbone = nn.Sequential(
            YoloConv(3, 16, 3, 2),
            YoloConv(16, 32, 3, 2),
            YoloC2f(32, 32, repeats=1),
            YoloConv(32, 64, 3, 2),
            YoloC2f(64, 64, repeats=2),
            YoloConv(64, 128, 3, 2),
            YoloC2f(128, 128, repeats=2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dropout = nn.Dropout(feature_dropout)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
            dropout=0.0,
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, attention_size),
            nn.Tanh(),
            nn.Linear(attention_size, 1),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, head_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_size, 1),
        )

        self.sigmoid = nn.Sigmoid()
        self.initialize_weights()

    def initialize_weights(self):
        for layer in self.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.kaiming_normal_(
                    layer.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )

            elif isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

        for name, parameter in self.lstm.named_parameters():
            if "weight_ih" in name:
                for gate in parameter.chunk(4, dim=0):
                    nn.init.xavier_uniform_(gate)

            elif "weight_hh" in name:
                for gate in parameter.chunk(4, dim=0):
                    nn.init.orthogonal_(gate)

            elif "bias" in name:
                nn.init.zeros_(parameter)

    def encode_frames(self, frames):
        features = self.backbone(frames)
        return self.pool(features).flatten(1)

    def forward_logits(self, videos, lengths):
        packed = pack_padded_sequence(
            videos,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        encoded = []

        for frames in packed.data.split(self.frame_chunk):
            if (
                self.training
                and self.checkpoint_cnn
                and torch.is_grad_enabled()
            ):
                # Reentrant checkpoint requires a gradient-enabled input.
                frames = frames.detach().requires_grad_(True)

                features = activation_checkpoint(
                    self.encode_frames,
                    frames,
                    use_reentrant=True,
                )
            else:
                features = self.encode_frames(frames)

            encoded.append(features)

        features = self.feature_dropout(
            torch.cat(encoded, dim=0)
        )

        packed_features = PackedSequence(
            features,
            packed.batch_sizes,
            packed.sorted_indices,
            packed.unsorted_indices,
        )

        packed_output, _ = self.lstm(packed_features)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
        )

        steps = torch.arange(
            output.size(1),
            device=output.device,
        )[None, :]

        valid = steps < lengths.to(output.device)[:, None]

        scores = self.attention(output).squeeze(-1)
        scores = scores.masked_fill(~valid, float("-inf"))

        attention = torch.softmax(scores, dim=1)
        pooled = (output * attention.unsqueeze(-1)).sum(dim=1)

        return self.classifier(pooled).squeeze(-1)

    def forward(self, videos, lengths):
        return self.sigmoid(self.forward_logits(videos, lengths))

In [12]:
model = YoloLSTMClassifier(
    hidden_size=CFG["hidden_size"],
    attention_size=CFG["attention_size"],
    head_size=CFG["head_size"],
    feature_dropout=CFG["feature_dropout"],
    dropout=CFG["dropout"],
    frame_chunk=CFG["frame_chunk"],
    checkpoint_cnn=CFG["checkpoint_cnn"],
).to(device)

assert not any(
    isinstance(layer, nn.BatchNorm2d)
    for layer in model.modules()
), "The model still contains BatchNorm."

gn_count = sum(
    isinstance(layer, nn.GroupNorm)
    for layer in model.modules()
)
assert gn_count == 20, f"Expected 20 GroupNorm layers, got {gn_count}"

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Device:", device, "| PyTorch:", torch.__version__)
print("GroupNorm layers:", gn_count)
print("CNN checkpointing:", model.checkpoint_cnn)
print(f"Trainable parameters: {parameter_count:,}")
print(f"FP32 parameter storage only: {parameter_count * 4 / 1024**2:.2f} MiB")

was_training = model.training
model.eval()

with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224, device=device)
    print("Input:", tuple(dummy.shape))

    for stage, layer in enumerate(model.backbone):
        dummy = layer(dummy)
        print(f"Backbone stage {stage}:", tuple(dummy.shape))

    print(
        "Pooled frame features:",
        tuple(model.pool(dummy).flatten(1).shape),
    )

model.train(was_training)
del dummy

Device: cuda | PyTorch: 2.11.0+cu130
GroupNorm layers: 20
CNN checkpointing: True
Trainable parameters: 406,354
FP32 parameter storage only: 1.55 MiB
Input: (1, 3, 224, 224)
Backbone stage 0: (1, 16, 112, 112)
Backbone stage 1: (1, 32, 56, 56)
Backbone stage 2: (1, 32, 56, 56)
Backbone stage 3: (1, 64, 28, 28)
Backbone stage 4: (1, 64, 28, 28)
Backbone stage 5: (1, 128, 14, 14)
Backbone stage 6: (1, 128, 14, 14)
Pooled frame features: (1, 128)


In [13]:
def check_new_model():
    print("PyTorch:", torch.__version__, "| Device:", device)

    # 1. Check autograd independently of the model.
    try:
        x = torch.ones(1, device=device, requires_grad=True)
        x.square().sum().backward()
        assert torch.allclose(x.grad, torch.full_like(x, 2.0))
    except Exception as error:
        raise RuntimeError(
            "Basic autograd failed. Restart the runtime "
            "and rerun cells in order."
        ) from error

    # 2. Verify that the latest model is active.
    assert type(model) is YoloLSTMClassifier, (
        "Rerun the model creation cell."
    )

    names = model.forward_logits.__func__.__code__.co_names
    assert "activation_checkpoint" in names, (
        "The latest model definition is not active."
    )

    assert not any(
        isinstance(m, nn.BatchNorm2d) for m in model.modules()
    )
    assert sum(
        isinstance(m, nn.GroupNorm) for m in model.modules()
    ) == 20

    # 3. Prepare two short clips for diagnosis only.
    samples = []

    for target in (0, 1):
        idx = int(np.flatnonzero(
            train_eval_ds.df["label"].to_numpy() == target
        )[0])

        frames, label = train_eval_ds[idx]

        assert len(frames) > 0 and torch.isfinite(frames).all()
        assert tuple(frames.shape[1:]) == (3, 224, 224)
        assert 0 <= frames.min() and frames.max() <= 1

        take = torch.linspace(
            0, len(frames) - 1, min(8, len(frames))
        ).long()

        samples.append((frames[take], label))

    videos, lengths, labels = collate_videos(samples)
    videos = videos.to(device)
    labels = labels.to(device)

    cuda_devices = (
        [torch.cuda.current_device()] if device.type == "cuda" else []
    )

    # Preserve the training RNG state.
    with torch.random.fork_rng(devices=cuda_devices):
        # Create a separate model; dropout is off only for this test.
        probe = YoloLSTMClassifier(
            hidden_size=CFG["hidden_size"],
            attention_size=CFG["attention_size"],
            head_size=CFG["head_size"],
            feature_dropout=0.0,
            dropout=0.0,
            frame_chunk=CFG["frame_chunk"],
            checkpoint_cnn=True,
        ).to(device)

        probe.load_state_dict(model.state_dict())
        probe.train()

        for enabled in (False, True):
            probe.checkpoint_cnn = enabled
            probe.zero_grad(set_to_none=True)

            logits = probe.forward_logits(videos, lengths)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits, labels
            )

            assert torch.isfinite(loss), "Non-finite loss."
            loss.backward()

            for name, parameter in probe.named_parameters():
                gradient = parameter.grad
                assert gradient is not None, name
                assert torch.isfinite(gradient).all(), name

            for name in ("backbone", "lstm", "attention", "classifier"):
                magnitude = sum(
                    p.grad.abs().sum().item()
                    for p in getattr(probe, name).parameters()
                )
                assert magnitude > 0, name

            if not enabled:
                reference = logits.detach().clone()
                gradients = {
                    name: p.grad.detach().clone()
                    for name, p in probe.named_parameters()
                }
            else:
                torch.testing.assert_close(
                    logits.detach(), reference,
                    atol=1e-4, rtol=1e-3,
                )

                for name, parameter in probe.named_parameters():
                    torch.testing.assert_close(
                        parameter.grad,
                        gradients[name],
                        atol=1e-4,
                        rtol=1e-3,
                        msg=name,
                    )

            print(
                "Backward OK | checkpoint:", enabled,
                flush=True,
            )

        probe.eval()
        with torch.no_grad():
            evaluated = probe.forward_logits(videos, lengths)

        torch.testing.assert_close(
            evaluated, reference,
            atol=1e-4, rtol=1e-3,
        )

    print(
        "CHECK PASSED: gradients, checkpointing, "
        "train/eval consistency."
    )


check_new_model()

PyTorch: 2.11.0+cu130 | Device: cuda
Backward OK | checkpoint: False
Backward OK | checkpoint: True
CHECK PASSED: gradients, checkpointing, train/eval consistency.


## ۵. یادگیری وزن‌ها

همهٔ پارامترها، شامل backbone، LSTM، attention و طبقه‌بند از صفر آموزش می‌بینند. loss از logit خام استفاده می‌کند؛ sigmoid فقط برای احتمال خروجی است.

AdamW با نرخ اولیهٔ 0.0003، weight decay برابر 0.0001 روی ماتریس‌های وزن و gradient clipping برابر ۱ استفاده می‌شود. Scheduler براساس validation loss نرخ را نصف می‌کند. آستانهٔ گزارش معیارها ثابت و برابر ۰٫۵ است.

Weight decay روی bias و پارامترهای normalization اعمال نمی‌شود. رفتار dropout و BatchNorm در ارزیابی به حالت `eval` تغییر می‌کند.

In [14]:
criterion = nn.BCEWithLogitsLoss()
decay_params, no_decay_params = [], []

for parameter in model.parameters():
    if parameter.ndim > 1:
        decay_params.append(parameter)
    else:
        no_decay_params.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": CFG["weight_decay"]},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=CFG["lr"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=CFG["lr_factor"],
    patience=CFG["lr_patience"], min_lr=CFG["min_lr"],
)

In [15]:
lr_trace = []


def train_one_epoch():
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0

    for i, (videos, lengths, labels) in enumerate(train_loader):
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model.forward_logits(videos, lengths)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")

        # Correct sample weighting for a partial final accumulation group.
        group_start = (i // CFG["accum_steps"]) * CFG["accum_steps"]
        group_samples = min(
            CFG["accum_steps"] * train_loader.batch_size,
            len(train_ds) - group_start * train_loader.batch_size,
        )
        (loss * labels.numel() / group_samples).backward()
        total_loss += loss.item() * labels.numel()
        if i == 0 or (i + 1) % CFG["log_every"] == 0 or i + 1 == len(train_loader):
            print(f"  Train batches: {i + 1}/{len(train_loader)} | "
                  f"batch loss: {loss.item():.4f}", flush=True)

        if (
            (i + 1) % CFG["accum_steps"] == 0
            or i + 1 == len(train_loader)
        ):
            nn.utils.clip_grad_norm_(
                model.parameters(), CFG["grad_clip"], error_if_nonfinite=True
            )
            lr_trace.append(float(optimizer.param_groups[0]["lr"]))
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    return total_loss / len(train_ds)


@torch.no_grad()
def evaluate(loader, name="Evaluation"):
    model.eval()
    total_loss, all_labels, all_probs = 0.0, [], []

    for i, (videos, lengths, labels) in enumerate(loader):
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model.forward_logits(videos, lengths)
        loss = criterion(logits, labels)
        probabilities = model.sigmoid(logits)
        total_loss += loss.item() * labels.numel()
        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probabilities.cpu().numpy().tolist())
        if i == 0 or (i + 1) % CFG["log_every"] == 0 or i + 1 == len(loader):
            print(f"  {name} batches: {i + 1}/{len(loader)}", flush=True)

    y, p = np.asarray(all_labels), np.asarray(all_probs)
    if not np.isfinite(p).all():
        raise RuntimeError("Non-finite predictions")
    pred = (p >= CFG["threshold"]).astype(int)
    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y, pred),
        "f1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "auc": roc_auc_score(y, p),
        "ap": average_precision_score(y, p),
    }
    # Plain Python floats keep weights_only=True checkpoint loading compatible.
    metrics = {key: float(value) for key, value in metrics.items()}
    return metrics, y, p

### بازیابی اجرای خودتان در صورت وجود

معماری، شناسهٔ تقسیم داده و تنظیمات آموزش کنترل می‌شوند. وضعیت optimizer، scheduler و تصادفی‌ها نیز بازیابی می‌شوند؛ برابری عددی دقیق بین سخت‌افزارها و نسخه‌های متفاوت PyTorch تضمین نمی‌شود.

In [16]:
def publish_file(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".uploading")
    try:
        shutil.copyfile(source, temporary)
        if temporary.stat().st_size != source.stat().st_size:
            raise IOError(f"Incomplete output copy: {destination}")
        temporary.replace(destination)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise


work_key = hashlib.sha256(str(OUT).encode()).hexdigest()[:12]
CHECKPOINT_WORK = LOCAL_PROJECT_DIR / "checkpoint_work" / work_key
CHECKPOINT_WORK.mkdir(parents=True, exist_ok=True)


def save_checkpoint(state, destination):
    local = CHECKPOINT_WORK / destination.name
    try:
        torch.save(state, local)
        publish_file(local, destination)
    finally:
        local.unlink(missing_ok=True)


def save_table(frame, destination):
    local = CHECKPOINT_WORK / destination.name
    try:
        frame.to_csv(local, index=False)
        publish_file(local, destination)
    finally:
        local.unlink(missing_ok=True)


def capture_random_states():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": (numpy_state[0], numpy_state[1].tolist(),
                  int(numpy_state[2]), int(numpy_state[3]), float(numpy_state[4])),
        "torch_cpu": torch.get_rng_state(),
        "loader": generator.get_state(),
        "cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
    }


def restore_random_states(states):
    random.setstate(states["python"])
    name, keys, position, has_gauss, cached_gauss = states["numpy"]
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), position, has_gauss, cached_gauss))
    torch.set_rng_state(states["torch_cpu"].cpu())
    generator.set_state(states["loader"].cpu())
    if device.type == "cuda" and states["cuda"]:
        torch.cuda.set_rng_state_all([state.cpu() for state in states["cuda"]])


history = []
lr_trace.clear()
best_key = (-float("inf"),) * 3
best_epoch = 0
bad_epochs = 0
start_epoch = 1
training_complete = False
best_path, last_path = OUT / "best.pt", OUT / "last.pt"

checkpoint_path = last_path if last_path.is_file() else best_path
if not CFG["resume"] and (last_path.is_file() or best_path.is_file()):
    raise FileExistsError("Choose a new out_dir for a fresh run, or set resume=True.")
if CFG["resume"] and checkpoint_path.is_file():
    restored = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if restored.get("checkpoint_schema") != 2:
        raise ValueError("This checkpoint is from an older format; use a new out_dir.")
    if restored["architecture"] != model.__class__.__name__ or restored["data_signature"] != data_signature:
        raise ValueError("Checkpoint architecture or dataset split differs; use a new out_dir.")
    locked_keys = [
        "seed", "batch_size", "accum_steps", "frame_chunk", "checkpoint_cnn",
        "hidden_size", "attention_size", "head_size", "feature_dropout", "dropout",
        "lr", "weight_decay", "grad_clip", "lr_factor", "lr_patience", "min_lr",
        "threshold", "min_epochs", "early_patience","selection_rule", "threshold_grid",
    ]
    for key in locked_keys:
        if restored["config"][key] != CFG[key]:
            raise ValueError(f"Resume setting changed: {key}. Use a new out_dir for a new experiment.")
    if not best_path.is_file():
        raise FileNotFoundError("The best.pt belonging to this run is missing from Drive.")
    model.load_state_dict(restored["model_state"])
    optimizer.load_state_dict(restored["optimizer_state"])
    scheduler.load_state_dict(restored["scheduler_state"])
    history = restored["history"]
    lr_trace[:] = restored["lr_trace"]
    best_key = tuple(restored["best_key"])
    best_epoch, bad_epochs = restored["best_epoch"], restored["bad_epochs"]
    start_epoch = restored["epoch"] + 1
    training_complete = restored["stopped_early"] or start_epoch > CFG["epochs"]
    restore_random_states(restored["random_states"])
    del restored
    print(f"Resuming YOUR training checkpoint: {checkpoint_path.name} | next epoch: {start_epoch}")
else:
    print("Fresh training: all weights initialized from scratch; no pretrained weights downloaded.")


Resuming YOUR training checkpoint: last.pt | next epoch: 13


## ۶. آموزش و ذخیره روی Drive

هر epoch شامل آموزش، ارزیابی train بدون آگمنتیشن و ارزیابی validation است. پیام شروع epoch و پیشرفت batchها نمایش داده می‌شوند. معیار انتخاب `best.pt` ابتدا F1 کلاس تصادف، سپس Accuracy و سپس Loss اعتبارسنجی است. scheduler بر اساس validation loss تنظیم می‌شود. حداقل ۱۵ و حداکثر ۴۰ epoch و patience=8 حفظ شده‌اند.

در پایان هر epoch، `last.pt` همراه optimizer، scheduler، تاریخچه و وضعیت مولدهای تصادفی ذخیره می‌شود. قطع اتصال می‌تواند کار epoch ناتمام را از دست بدهد؛ ادامه از epoch تکمیل‌شده است. برای ادامه پس از حذف runtime، GPU و Drive را وصل کنید و کل نوت‌بوک را از ابتدا با همان تنظیمات اجرا کنید.


In [17]:
def threshold_metrics(y, p, threshold):
    pred = (np.asarray(p) >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "macro_f1": float(f1_score(
            y, pred, average="macro", zero_division=0
        )),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(
            y, pred, zero_division=0
        )),
        "recall": float(recall_score(
            y, pred, zero_division=0
        )),
        "predicted_positive": int(pred.sum()),
    }


def select_f1_threshold(y, p):
    return max(
        (
            threshold_metrics(y, p, threshold)
            for threshold in CFG["threshold_grid"]
        ),
        key=lambda row: (
            row["f1"],
            row["macro_f1"],
            -abs(row["threshold"] - 0.5),
        ),
    )

In [18]:
if training_complete:
    print(
        "This saved run has already finished. "
        "Plot its results, or choose a new out_dir for a new run."
    )

for epoch in range(start_epoch, CFG["epochs"] + 1):
    if training_complete:
        break

    started = time.time()

    print(
        f"\nEpoch {epoch:02d}/{CFG['epochs']} | Training started",
        flush=True,
    )

    # 1. Update model weights.
    update_loss = train_one_epoch()

    # 2. Evaluate training data without augmentation.
    train_metrics, _, _ = evaluate(
        train_eval_loader,
        name="Clean train",
    )

    # 3. Evaluate validation data.
    val_metrics, y_val, p_val = evaluate(
        val_loader,
        name="Validation",
    )

    # 4. Select threshold using validation data only.
    chosen = select_f1_threshold(y_val, p_val)

    print(
        f"Selected validation F1: {chosen['f1']:.4f} "
        f"| threshold: {chosen['threshold']:.2f}",
        flush=True,
    )

    # 5. Record metrics.
    row = {
        "epoch": epoch,
        "update_loss": float(update_loss),
        "lr": float(optimizer.param_groups[0]["lr"]),
    }

    row.update({
        f"train_{k}": v
        for k, v in train_metrics.items()
    })

    row.update({
        f"val_{k}": v
        for k, v in val_metrics.items()
    })

    row["val_tuned_f1"] = chosen["f1"]
    row["val_threshold"] = chosen["threshold"]

    history.append(row)

    # Scheduler uses validation loss.
    scheduler.step(val_metrics["loss"])

    # 6. Select best epoch:
    # higher tuned validation F1, then higher AUC, then lower loss.
    current_key = (
        chosen["f1"],
        val_metrics["auc"],
        -val_metrics["loss"],
    )

    improved = current_key > best_key

    if improved:
        best_key = current_key
        best_epoch = epoch
        bad_epochs = 0
    else:
        bad_epochs += 1

    should_stop = (
        epoch >= CFG["min_epochs"]
        and bad_epochs >= CFG["early_patience"]
    )

    # 7. Save weights together with their selected threshold.
    state = {
        "checkpoint_schema": 2,
        "architecture": model.__class__.__name__,
        "data_signature": data_signature,

        "epoch": epoch,
        "model_state": model.state_dict(),
        "decision_threshold": chosen["threshold"],
        "selection_metrics": dict(chosen),

        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),

        "config": dict(CFG),
        "val_metrics": dict(val_metrics),

        "history": list(history),
        "lr_trace": list(lr_trace),

        "best_key": best_key,
        "best_epoch": best_epoch,
        "bad_epochs": bad_epochs,

        "stopped_early": bool(should_stop),
        "random_states": capture_random_states(),
    }

    if improved:
        save_checkpoint(state, OUT / "best.pt")

    save_checkpoint(state, OUT / "last.pt")
    del state

    # 8. Save training logs.
    save_table(
        pd.DataFrame(history),
        OUT / "history.csv",
    )

    save_table(
        pd.DataFrame({
            "step": np.arange(1, len(lr_trace) + 1),
            "lr": lr_trace,
        }),
        OUT / "learning_rates.csv",
    )

    start_epoch = epoch + 1

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | "
        f"Acc {train_metrics['accuracy']:.3f}/{val_metrics['accuracy']:.3f} | "
        f"F1@{CFG['threshold']:.2f} "
        f"{train_metrics['f1']:.3f}/{val_metrics['f1']:.3f} | "
        f"Val AUC {val_metrics['auc']:.3f} | "
        f"Tuned val F1 {chosen['f1']:.3f}@{chosen['threshold']:.2f} | "
        f"LR {row['lr']:.2e} | "
        f"{time.time() - started:.0f}s"
        + (" | BEST" if improved else ""),
        flush=True,
    )

    if should_stop:
        training_complete = True
        print("Early stopping.", flush=True)
        break

if start_epoch > CFG["epochs"]:
    training_complete = True


Epoch 13/15 | Training started
  Train batches: 1/340 | batch loss: 0.4229
  Train batches: 10/340 | batch loss: 0.5917
  Train batches: 20/340 | batch loss: 0.7084
  Train batches: 30/340 | batch loss: 0.3621
  Train batches: 40/340 | batch loss: 1.4313
  Train batches: 50/340 | batch loss: 1.0747
  Train batches: 60/340 | batch loss: 0.3575
  Train batches: 70/340 | batch loss: 1.1589
  Train batches: 80/340 | batch loss: 0.7660
  Train batches: 90/340 | batch loss: 0.3779
  Train batches: 100/340 | batch loss: 0.2410
  Train batches: 110/340 | batch loss: 0.5122
  Train batches: 120/340 | batch loss: 0.4345
  Train batches: 130/340 | batch loss: 1.0511
  Train batches: 140/340 | batch loss: 0.3565
  Train batches: 150/340 | batch loss: 1.1549
  Train batches: 160/340 | batch loss: 0.4783
  Train batches: 170/340 | batch loss: 0.4814
  Train batches: 180/340 | batch loss: 0.4943
  Train batches: 190/340 | batch loss: 0.2860
  Train batches: 200/340 | batch loss: 0.6520
  Train batch

In [ ]:
report_path = OUT / "test_metrics.json"

if report_path.exists():
    print("Previously saved final test result:")
    print(report_path.read_text())

else:
    assert training_complete, "Finish training first."

    chosen_checkpoint = torch.load(
        OUT / "best.pt",
        map_location="cpu",
        weights_only=True,
    )

    assert (
        chosen_checkpoint["config"]["selection_rule"]
        == CFG["selection_rule"]
    )
    assert chosen_checkpoint["data_signature"] == data_signature

    model.load_state_dict(chosen_checkpoint["model_state"])

    chosen_threshold = float(
        chosen_checkpoint["decision_threshold"]
    )
    chosen_epoch = int(chosen_checkpoint["epoch"])
    del chosen_checkpoint

    test_ds = AccidentVideoDataset(
        SOURCE_FRAME_DIR / CFG["test_csv"],
        SOURCE_FRAME_DIR,
        augment=False,
    )

    assert len(test_ds) == 30
    assert test_ds.df["label"].value_counts().to_dict() == {0: 15, 1: 15}

    all_ids = pd.concat([
        train_ds.df, val_ds.df, test_ds.df
    ])["video_path"]

    assert len(all_ids) == 430 and all_ids.is_unique

    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_videos,
        generator=torch.Generator().manual_seed(CFG["seed"]),
    )

    test_metrics, y_test, p_test = evaluate(
        test_loader, name="FINAL TEST"
    )

    test_metrics.update(
        threshold_metrics(y_test, p_test, chosen_threshold)
    )
    test_metrics["epoch"] = chosen_epoch

    predictions = test_ds.df[["video_path", "label"]].copy()
    predictions["probability"] = p_test
    predictions["prediction"] = (
        p_test >= chosen_threshold
    ).astype(int)

    save_table(predictions, OUT / "test_predictions.csv")

    local_report = CHECKPOINT_WORK / report_path.name
    try:
        local_report.write_text(
            json.dumps(test_metrics, indent=2)
        )
        publish_file(local_report, report_path)
    finally:
        local_report.unlink(missing_ok=True)

    print(json.dumps(test_metrics, indent=2))

  FINAL TEST batches: 1/30
  FINAL TEST batches: 10/30
  FINAL TEST batches: 20/30
  FINAL TEST batches: 30/30
{
  "loss": 0.5994035800298055,
  "accuracy": 0.6333333333333333,
  "f1": 0.717948717948718,
  "precision": 0.5833333333333334,
  "recall": 0.9333333333333333,
  "auc": 0.7955555555555556,
  "ap": 0.8254737734832632,
  "threshold": 0.4,
  "macro_f1": 0.5970695970695971,
  "predicted_positive": 24,
  "epoch": 12
}


## ۷. منحنی‌های یادگیری و بررسی overfitting / underfitting

- افت train loss همراه با افزایش مداوم validation loss و بیشترشدن فاصلهٔ عملکرد، نشانهٔ احتمالی **overfitting** است.
- ضعیف‌ماندن عملکرد هر دو مجموعه می‌تواند نشانهٔ **underfitting** باشد؛ مشکل داده یا نرخ یادگیری نامناسب هم می‌تواند همین ظاهر را بسازد.
- نزدیک‌بودن دو منحنی به‌تنهایی کافی نیست؛ سطح F1 و accuracy را هم ببینید.
- خط عمودی، epoch انتخاب‌شده با معیار checkpoint را نشان می‌دهد.
- نرخ یادگیری بر حسب optimizer step رسم می‌شود، نه تعداد batchها.

In [ ]:
history_df = pd.DataFrame(history)
epochs = history_df["epoch"]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

curve_specs = [
    (axes[0, 0], "loss", "Loss — clean evaluation"),
    (axes[0, 1], "accuracy", "Accuracy"),
    (axes[0, 2], "f1", f"F1 at threshold {CFG['threshold']}"),
]
for ax, metric, title in curve_specs:
    ax.plot(epochs, history_df[f"train_{metric}"], label="Train")
    ax.plot(epochs, history_df[f"val_{metric}"], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Selected epoch")
    ax.set(title=title, xlabel="Epoch")
    ax.legend()

axes[1, 0].plot(np.arange(1, len(lr_trace) + 1), lr_trace)
axes[1, 0].set(
    title="Learning rate curve", xlabel="Optimizer step", ylabel="Learning rate"
)
axes[1, 0].set_yscale("log")

gap = history_df["train_accuracy"] - history_df["val_accuracy"]
axes[1, 1].plot(epochs, gap, color="purple")
axes[1, 1].axhline(0, color="gray", linestyle="--")
axes[1, 1].set(
    title="Generalization gap", xlabel="Epoch",
    ylabel="Train accuracy − validation accuracy",
)

axes[1, 2].plot(epochs, history_df["val_precision"], label="Precision")
axes[1, 2].plot(epochs, history_df["val_recall"], label="Recall")
axes[1, 2].set(title="Validation precision / recall", xlabel="Epoch")
axes[1, 2].legend()

for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "learning_curves.png", dpi=160)
plt.show()

## ۸. بهترین checkpoint و نمودارهای اعتبارسنجی

این نتایج مربوط به **validation** هستند و برای انتخاب وزن استفاده شده‌اند؛ نتیجهٔ test مستقل محسوب نمی‌شوند.

ROC، منحنی precision–recall و confusion matrix رسم می‌شوند. فایل پیش‌بینی‌ها کمک می‌کند ویدئوهای اشتباه بررسی شوند.

In [ ]:
saved = torch.load(OUT / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(saved["model_state"])
model.eval()

best_metrics, y_val, p_val = evaluate(val_loader)
decision_threshold = float(saved["decision_threshold"])
pred_val = (p_val >= decision_threshold).astype(int)
best_metrics.update(threshold_metrics(y_val, p_val, decision_threshold))
print("Selected epoch:", saved["epoch"])
print("VALIDATION metrics — not an independent test result:")
print(pd.Series(best_metrics).round(4))

predictions = val_ds.df[["video_path", "label"]].copy()
predictions["probability"] = p_val
predictions["prediction"] = pred_val
predictions["correct"] = (
    predictions["label"] == predictions["prediction"]
)
predictions.to_csv(OUT / "validation_predictions.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fpr, tpr, _ = roc_curve(y_val, p_val)
axes[0].plot(fpr, tpr, label=f"AUC = {best_metrics['auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(
    title="Validation ROC", xlabel="False positive rate",
    ylabel="True positive rate",
)
axes[0].legend()

precision, recall, _ = precision_recall_curve(y_val, p_val)
axes[1].plot(recall, precision, label=f"AP = {best_metrics['ap']:.3f}")
axes[1].axhline(
    y_val.mean(), color="gray", linestyle="--", label="Class prevalence"
)
axes[1].set(
    title="Validation precision–recall", xlabel="Recall", ylabel="Precision"
)
axes[1].legend()

ConfusionMatrixDisplay.from_predictions(
    y_val, pred_val, labels=[0, 1],
    display_labels=["No accident", "Accident"],
    cmap="Blues", colorbar=False, ax=axes[2],
)
axes[2].set_title("Validation confusion matrix")
fig.tight_layout()
fig.savefig(OUT / "validation_diagnostics.png", dpi=160)
plt.show()

## ۹. نمونهٔ پیش‌بینی با خروجی Sigmoid

این سلول روی یک batch از validation نشان می‌دهد که خروجی عمومی مدل یک احتمال بین صفر و یک است.

In [ ]:
model.eval()
videos, lengths, labels = next(iter(val_loader))
with torch.no_grad():
    probabilities = model(videos.to(device), lengths)
    predicted_labels = (
        probabilities >= decision_threshold
    ).long()

print("True labels:", labels.long().tolist())
print("Accident probabilities:", probabilities.cpu().tolist())
print("Predicted labels:", predicted_labels.cpu().tolist())

## خروجی‌ها

در `CFG["out_dir"]` روی Drive:

- `best.pt`: checkpoint منتخب با بهترین F1 اعتبارسنجی، سپس Accuracy و Loss.
- `last.pt`: آخرین epoch تکمیل‌شده برای ادامهٔ آموزش خودتان.
- `history.csv` و `learning_rates.csv`: تاریخچه و نرخ یادگیری.
- `learning_curves.png`: منحنی‌های Loss، Accuracy، F1، نرخ یادگیری و فاصلهٔ عملکرد train/validation.
- `validation_diagnostics.png`: نمودار ROC، PR و ماتریس خطا.
- `validation_predictions.csv`: احتمال و برچسب پیش‌بینی‌شدهٔ هر ویدئوی اعتبارسنجی.

فایل‌ها ابتدا محلی کامل می‌شوند و checkpointها با نام موقت به Drive کپی می‌شوند. فایل تکمیل‌شدهٔ قبلی تا پایان کپی جایگزین نمی‌شود. بازیابی به فایل‌های تکمیل‌شدهٔ موجود روی Drive وابسته است. `best.pt` و `last.pt` مربوط به یک اجرا را کنار هم نگه دارید. ادامه از وزن‌های اجرای قبلی خودتان است، نه وزن آمادهٔ خارجی.

معماری و تعریف Dataset/augmentation قبلی حفظ شده‌اند. هیچ عملکرد بهتری صرفاً به‌خاطر انتقال به Colab تضمین نمی‌شود. نتایج نهایی مربوط به validation مورد استفاده در انتخاب checkpoint هستند؛ test مستقل نیستند. F1 برای کلاس تصادف با label=1 گزارش می‌شود.

منابع: [راهنمای ذخیره و بازیابی PyTorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)، [Colab و Drive](https://research.google.com/colaboratory/faq.html)، [Packed sequences](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html)، [BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html).
